# Notebook 03 — Baseline Predictor Table and Missingness Assessment

**Project:** Machine Learning-Based Prediction of Parkinson's Disease Progression Using PPMI Data  
**Stage:** Predictor dataset construction and quality-control review  
**Primary outcome file:** Output from Notebook 02  
**Primary cohort:** Parkinson's Disease participants only  
**Primary follow-up window:** Recommended by Notebook 02, expected `V06`

---

## Objective

This notebook constructs a baseline-only predictor table for the PPMI Parkinson's disease progression project. It does **not** perform machine learning modeling.

The goals are to:

1. Load the primary analytic cohort created by Notebook 02.
2. Verify the selected primary outcome and follow-up window.
3. Define candidate baseline predictors only.
4. Exclude follow-up variables and outcome-derived variables to prevent data leakage.
5. Create a predictor dictionary.
6. Assess missingness.
7. Create a clean feature matrix for the next notebook.
8. Save all outputs for scientific review.

---

## Scientific Background

For a clinically valid predictive model, predictors must be measured at baseline or before the prediction time. Variables measured after baseline, follow-up outcome values, or variables derived from the outcome must not be used as model inputs.

The current project focuses on predicting motor progression using MDS-UPDRS Part III change from baseline to the selected follow-up visit. Therefore, follow-up Part III values, delta values, annualized delta values, and rapid-progression labels are outcomes, not predictors.

## Dataset Verification

This notebook expects the following file created by Notebook 02:

`MyDrive/PPMI_PD_Progression/outputs/notebook_02_cohort_outcome/11_primary_analytic_cohort_recommended_window.csv`

Expected key columns include:

- `PATNO`
- `baseline_NP3TOT`
- `followup_NP3TOT`
- `delta_NP3TOT`
- `annualized_delta_NP3TOT`
- `rapid_progression_q75`
- baseline clinical variables such as MoCA, UPSIT, MDS-UPDRS Part I/II, vitals, and diagnosis-history variables.

If this file is missing, rerun Notebook 02 first.

In [ ]:
# ============================================================
# 01. Environment setup
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("Python environment is ready.")
print("pandas:", pd.__version__)

In [ ]:
# ============================================================
# 02. Mount Google Drive
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print("Not running in Google Colab, or Google Drive mount failed.")
    print("Error:", e)

In [ ]:
# ============================================================
# 03. Define project paths
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

NOTEBOOK02_DIR = PROJECT_DIR / "outputs" / "notebook_02_cohort_outcome"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "notebook_03_baseline_predictors"

COHORT_FILE = NOTEBOOK02_DIR / "11_primary_analytic_cohort_recommended_window.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("NOTEBOOK02_DIR:", NOTEBOOK02_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("COHORT_FILE exists:", COHORT_FILE.exists())

if not COHORT_FILE.exists():
    raise FileNotFoundError(
        f"Required Notebook 02 output file not found:\n{COHORT_FILE}\n"
        "Please rerun Notebook 02 or confirm that the project path is correct."
    )

In [ ]:
# ============================================================
# 04. Load primary analytic cohort from Notebook 02
# ============================================================

cohort = pd.read_csv(COHORT_FILE)

print("Cohort shape:", cohort.shape)
display(cohort.head())
print("\nColumns:")
print(list(cohort.columns))

In [ ]:
# ============================================================
# 05. Verify required outcome columns
# ============================================================

required_outcome_cols = [
    "PATNO",
    "baseline_event",
    "followup_event",
    "baseline_NP3TOT",
    "followup_NP3TOT",
    "delta_NP3TOT",
    "annualized_delta_NP3TOT",
    "rapid_progression_q75",
]

required_check = []
for col in required_outcome_cols:
    required_check.append({
        "column": col,
        "present": col in cohort.columns,
        "non_missing_n": int(cohort[col].notna().sum()) if col in cohort.columns else 0,
        "missing_n": int(cohort[col].isna().sum()) if col in cohort.columns else None,
    })

required_check_df = pd.DataFrame(required_check)
display(required_check_df)

if not required_check_df["present"].all():
    missing = required_check_df.loc[~required_check_df["present"], "column"].tolist()
    raise ValueError(f"Missing required columns: {missing}")

required_check_df.to_csv(OUTPUT_DIR / "01_required_outcome_column_check.csv", index=False)

In [ ]:
# ============================================================
# 06. Summarize primary outcome and follow-up window
# ============================================================

outcome_summary = {
    "n_rows": len(cohort),
    "n_unique_PATNO": cohort["PATNO"].nunique(),
    "baseline_events": "; ".join(map(str, sorted(cohort["baseline_event"].dropna().unique()))),
    "followup_events": "; ".join(map(str, sorted(cohort["followup_event"].dropna().unique()))),
    "rapid_progression_positive_n": int((cohort["rapid_progression_q75"] == 1).sum()),
    "rapid_progression_negative_n": int((cohort["rapid_progression_q75"] == 0).sum()),
    "rapid_progression_positive_percent": float((cohort["rapid_progression_q75"] == 1).mean() * 100),
    "delta_NP3TOT_mean": float(cohort["delta_NP3TOT"].mean()),
    "delta_NP3TOT_median": float(cohort["delta_NP3TOT"].median()),
    "annualized_delta_NP3TOT_mean": float(cohort["annualized_delta_NP3TOT"].mean()),
    "annualized_delta_NP3TOT_median": float(cohort["annualized_delta_NP3TOT"].median()),
    "annualized_delta_NP3TOT_q75": float(cohort["annualized_delta_NP3TOT"].quantile(0.75)),
}

outcome_summary_df = pd.DataFrame([outcome_summary])
display(outcome_summary_df)
outcome_summary_df.to_csv(OUTPUT_DIR / "02_outcome_distribution_summary.csv", index=False)

## Code — Data Leakage Prevention

The next cell separates variables into:

1. **ID columns**
2. **Outcome columns**
3. **Follow-up columns**
4. **Baseline predictor candidates**
5. **Columns excluded from modeling because they are outcome-derived, follow-up-derived, or administrative**

The guiding rule is:

> A predictor must be available at baseline or before baseline. It must not contain follow-up information or be derived from the outcome.

In [ ]:
# ============================================================
# 07. Define leakage columns and candidate baseline predictors
# ============================================================

id_cols = ["PATNO"]

# Outcome or outcome-derived variables. These must NOT be used as predictors.
outcome_cols = [
    "followup_NP3TOT",
    "delta_NP3TOT",
    "annualized_delta_NP3TOT",
    "rapid_progression_q75",
    "any_motor_worsening",
]

# Follow-up columns. These must NOT be used as baseline predictors.
followup_cols = [c for c in cohort.columns if c.startswith("followup_")]

# Other columns related to the constructed outcome-pair eligibility or future exam state.
pair_or_eligibility_cols = [
    "primary_exam_state_eligible",
    "sensitivity_exam_state_eligible",
    "on_exam_in_pair",
]

# Administrative or duplicated baseline-table versions that should not be used directly.
administrative_cols = [
    "COHORT_baseline_table",
    "COHORT_DEFINITION_baseline_table",
    "ENROLL_AGE_baseline_table",
    "ENRLLRRK2_baseline_table",
    "ENRLGBA_baseline_table",
    "ENRLSNCA_baseline_table",
    "ENRLPRKN_baseline_table",
    "ENRLRBD_baseline_table",
    "ENRLHPSM_baseline_table",
    "baseline_NP3TOT_baseline_table",
]

# Candidate baseline predictors manually selected from Notebook 02 cohort.
candidate_predictor_cols = [
    # Demographic / enrollment
    "ENROLL_AGE",

    # Genetic / enrollment subgroup indicators
    "ENRLLRRK2",
    "ENRLGBA",
    "ENRLSNCA",
    "ENRLPRKN",
    "ENRLRBD",
    "ENRLHPSM",

    # Baseline motor and non-motor severity
    "baseline_NP3TOT",
    "baseline_NHY",
    "part1_NP1RTOT",
    "part1p_NP1PTOT",
    "part2_NP2PTOT",

    # Cognition and olfaction
    "moca_MCATOT",
    "upsit_TOTAL_CORRECT",

    # Vitals / anthropometrics
    "vitals_WGTKG",
    "vitals_HTCM",
    "vitals_SYSSUP",
    "vitals_DIASUP",
    "vitals_HRSUP",

    # Diagnosis history / clinical presentation
    "pddx_DXTREMOR",
    "pddx_DXRIGID",
    "pddx_DXBRADY",
    "pddx_DXPOSINS",
    "pddx_DOMSIDE",

    # Primary diagnosis variables
    "primdiag_PRIMDIAG",
    "primdiag_DXLVL",
]

# Keep only columns that are actually present.
present_candidate_predictors = [c for c in candidate_predictor_cols if c in cohort.columns]
missing_candidate_predictors = [c for c in candidate_predictor_cols if c not in cohort.columns]

print("Candidate predictors requested:", len(candidate_predictor_cols))
print("Candidate predictors present:", len(present_candidate_predictors))
print("Candidate predictors missing:", missing_candidate_predictors)

excluded_cols = sorted(set(
    id_cols
    + outcome_cols
    + followup_cols
    + pair_or_eligibility_cols
    + administrative_cols
))

leakage_table = pd.DataFrame({
    "column": excluded_cols,
    "reason": [
        "ID/outcome/follow-up/eligibility/admin column excluded from baseline predictor matrix"
        for _ in excluded_cols
    ]
})

display(leakage_table.head(30))
leakage_table.to_csv(OUTPUT_DIR / "03_leakage_exclusion_table.csv", index=False)

In [ ]:
# ============================================================
# 08. Add derived baseline predictors
# ============================================================

feature_df = cohort[id_cols + present_candidate_predictors + ["rapid_progression_q75", "delta_NP3TOT", "annualized_delta_NP3TOT"]].copy()

# BMI from baseline weight and height.
if {"vitals_WGTKG", "vitals_HTCM"}.issubset(feature_df.columns):
    height_m = feature_df["vitals_HTCM"] / 100
    feature_df["derived_BMI"] = feature_df["vitals_WGTKG"] / (height_m ** 2)
    # Flag biologically implausible values as missing; this is conservative QC, not imputation.
    feature_df.loc[(feature_df["derived_BMI"] < 12) | (feature_df["derived_BMI"] > 70), "derived_BMI"] = np.nan

# Parse dates if possible for diagnosis duration variables.
def parse_ppmi_date(x):
    if pd.isna(x):
        return pd.NaT
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "nat", "none"}:
        return pd.NaT
    # PPMI files may use YYYY-MM-DD or MM/YYYY.
    for fmt in ("%Y-%m-%d", "%m/%Y", "%Y/%m", "%m/%d/%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except Exception:
            pass
    return pd.to_datetime(x, errors="coerce")

if "baseline_date" in cohort.columns:
    baseline_date = cohort["baseline_date"].apply(parse_ppmi_date)
    if "pddx_PDDXDT" in cohort.columns:
        pddx_date = cohort["pddx_PDDXDT"].apply(parse_ppmi_date)
        feature_df["derived_years_since_PD_diagnosis"] = (baseline_date - pddx_date).dt.days / 365.25
        feature_df.loc[
            (feature_df["derived_years_since_PD_diagnosis"] < -1) |
            (feature_df["derived_years_since_PD_diagnosis"] > 60),
            "derived_years_since_PD_diagnosis"
        ] = np.nan

    if "pddx_SXDT" in cohort.columns:
        sx_date = cohort["pddx_SXDT"].apply(parse_ppmi_date)
        feature_df["derived_years_since_symptom_onset"] = (baseline_date - sx_date).dt.days / 365.25
        feature_df.loc[
            (feature_df["derived_years_since_symptom_onset"] < -1) |
            (feature_df["derived_years_since_symptom_onset"] > 80),
            "derived_years_since_symptom_onset"
        ] = np.nan

print("Feature dataframe shape:", feature_df.shape)
display(feature_df.head())

In [ ]:
# ============================================================
# 09. Build predictor dictionary
# ============================================================

predictor_cols_final = [
    c for c in feature_df.columns
    if c not in ["PATNO", "rapid_progression_q75", "delta_NP3TOT", "annualized_delta_NP3TOT"]
]

predictor_dictionary_rows = []
for col in predictor_cols_final:
    if col.startswith("derived_"):
        source = "Derived from baseline variables"
    elif col.startswith("vitals_"):
        source = "Baseline vitals / anthropometrics"
    elif col.startswith("moca_"):
        source = "Baseline or screening cognition"
    elif col.startswith("upsit_"):
        source = "Baseline or screening olfaction"
    elif col.startswith("part"):
        source = "Baseline MDS-UPDRS"
    elif col.startswith("pddx_"):
        source = "Baseline PD diagnosis history"
    elif col.startswith("primdiag_"):
        source = "Primary research diagnosis"
    elif col.startswith("ENRL"):
        source = "Enrollment/subgroup characteristic"
    elif col.startswith("baseline_"):
        source = "Baseline motor examination"
    else:
        source = "Participant status / baseline characteristic"

    predictor_dictionary_rows.append({
        "predictor": col,
        "source_group": source,
        "dtype": str(feature_df[col].dtype),
        "non_missing_n": int(feature_df[col].notna().sum()),
        "missing_n": int(feature_df[col].isna().sum()),
        "missing_percent": float(feature_df[col].isna().mean() * 100),
        "used_in_notebook_03_feature_matrix": True,
        "note": "Baseline-only candidate predictor; no imputation or modeling performed in Notebook 03."
    })

predictor_dictionary = pd.DataFrame(predictor_dictionary_rows).sort_values("missing_percent")
display(predictor_dictionary)

predictor_dictionary.to_csv(OUTPUT_DIR / "04_candidate_predictor_dictionary.csv", index=False)

In [ ]:
# ============================================================
# 10. Missingness summary
# ============================================================

missingness_summary = predictor_dictionary[[
    "predictor", "source_group", "non_missing_n", "missing_n", "missing_percent"
]].copy()

# Recommended status based on missingness. These thresholds are provisional for QC review.
def missingness_status(pct):
    if pct == 0:
        return "Complete"
    elif pct <= 10:
        return "Low missingness"
    elif pct <= 30:
        return "Moderate missingness"
    elif pct <= 50:
        return "High missingness - consider sensitivity or careful imputation"
    else:
        return "Very high missingness - do not use in primary model without justification"

missingness_summary["missingness_status"] = missingness_summary["missing_percent"].apply(missingness_status)

display(missingness_summary)
missingness_summary.to_csv(OUTPUT_DIR / "05_predictor_missingness_summary.csv", index=False)

In [ ]:
# ============================================================
# 11. Outcome distribution table
# ============================================================

outcome_distribution = pd.DataFrame({
    "metric": [
        "n",
        "rapid_progression_positive_n",
        "rapid_progression_negative_n",
        "rapid_progression_positive_percent",
        "delta_NP3TOT_mean",
        "delta_NP3TOT_median",
        "delta_NP3TOT_q25",
        "delta_NP3TOT_q75",
        "annualized_delta_mean",
        "annualized_delta_median",
        "annualized_delta_q25",
        "annualized_delta_q75",
    ],
    "value": [
        len(feature_df),
        int((feature_df["rapid_progression_q75"] == 1).sum()),
        int((feature_df["rapid_progression_q75"] == 0).sum()),
        float((feature_df["rapid_progression_q75"] == 1).mean() * 100),
        float(feature_df["delta_NP3TOT"].mean()),
        float(feature_df["delta_NP3TOT"].median()),
        float(feature_df["delta_NP3TOT"].quantile(0.25)),
        float(feature_df["delta_NP3TOT"].quantile(0.75)),
        float(feature_df["annualized_delta_NP3TOT"].mean()),
        float(feature_df["annualized_delta_NP3TOT"].median()),
        float(feature_df["annualized_delta_NP3TOT"].quantile(0.25)),
        float(feature_df["annualized_delta_NP3TOT"].quantile(0.75)),
    ]
})

display(outcome_distribution)
outcome_distribution.to_csv(OUTPUT_DIR / "06_outcome_distribution_table.csv", index=False)

In [ ]:
# ============================================================
# 12. Create feature matrices
# ============================================================

# Full candidate feature matrix includes all candidate predictors regardless of missingness.
full_feature_matrix = feature_df[["PATNO"] + predictor_cols_final + ["rapid_progression_q75", "delta_NP3TOT", "annualized_delta_NP3TOT"]].copy()

# Low/moderate missingness feature matrix keeps predictors with <=30% missingness.
low_missing_predictors = missingness_summary.loc[
    missingness_summary["missing_percent"] <= 30, "predictor"
].tolist()

low_missing_feature_matrix = feature_df[["PATNO"] + low_missing_predictors + ["rapid_progression_q75", "delta_NP3TOT", "annualized_delta_NP3TOT"]].copy()

print("Full feature matrix shape:", full_feature_matrix.shape)
print("Low/moderate missingness feature matrix shape:", low_missing_feature_matrix.shape)
print("Low/moderate missingness predictors:", low_missing_predictors)

full_feature_matrix.to_csv(OUTPUT_DIR / "07_feature_matrix_all_candidate_predictors.csv", index=False)
low_missing_feature_matrix.to_csv(OUTPUT_DIR / "08_feature_matrix_predictors_missingness_le_30pct.csv", index=False)

In [ ]:
# ============================================================
# 13. Baseline source-event audit
# ============================================================

source_event_cols = [c for c in cohort.columns if c.endswith("_baseline_source_event")]
source_event_summary_rows = []

for col in source_event_cols:
    counts = cohort[col].value_counts(dropna=False).reset_index()
    counts.columns = ["source_event", "n"]
    counts.insert(0, "variable_group", col)
    source_event_summary_rows.append(counts)

if source_event_summary_rows:
    source_event_summary = pd.concat(source_event_summary_rows, ignore_index=True)
else:
    source_event_summary = pd.DataFrame(columns=["variable_group", "source_event", "n"])

display(source_event_summary)
source_event_summary.to_csv(OUTPUT_DIR / "09_baseline_source_event_audit.csv", index=False)

In [ ]:
# ============================================================
# 14. Quality-control checklist
# ============================================================

qc_rows = []

def add_qc(item, passed, detail):
    qc_rows.append({
        "qc_item": item,
        "status": "PASS" if passed else "REVIEW",
        "detail": detail
    })

add_qc(
    "Notebook 02 analytic cohort loaded",
    len(cohort) > 0,
    f"Rows: {len(cohort):,}; unique PATNO: {cohort['PATNO'].nunique():,}"
)

add_qc(
    "Outcome label available",
    "rapid_progression_q75" in cohort.columns and cohort["rapid_progression_q75"].notna().all(),
    f"Non-missing rapid progression labels: {cohort['rapid_progression_q75'].notna().sum():,}"
)

add_qc(
    "No duplicate PATNO in feature matrix",
    full_feature_matrix["PATNO"].is_unique,
    f"Duplicate PATNO count: {full_feature_matrix['PATNO'].duplicated().sum():,}"
)

add_qc(
    "Follow-up columns excluded from predictor matrix",
    not any(c.startswith("followup_") for c in predictor_cols_final),
    "No predictor starts with followup_."
)

add_qc(
    "Outcome-derived columns excluded from predictors",
    not any(c in predictor_cols_final for c in ["delta_NP3TOT", "annualized_delta_NP3TOT", "rapid_progression_q75", "any_motor_worsening"]),
    "Outcome and derived outcome columns are retained only as targets/secondary outcomes."
)

add_qc(
    "Candidate predictors available",
    len(predictor_cols_final) >= 10,
    f"Candidate predictor count: {len(predictor_cols_final)}"
)

add_qc(
    "Low/moderate missingness matrix available",
    len(low_missing_predictors) >= 5,
    f"Predictors with <=30% missingness: {len(low_missing_predictors)}"
)

add_qc(
    "No imputation performed",
    True,
    "Notebook 03 performs feature inventory and missingness assessment only."
)

add_qc(
    "No ML modeling performed",
    True,
    "Notebook 03 does not train or evaluate any machine learning model."
)

qc_checklist = pd.DataFrame(qc_rows)
display(qc_checklist)
qc_checklist.to_csv(OUTPUT_DIR / "10_quality_control_checklist.csv", index=False)

## Scientific Interpretation

At this stage, the project has a defined PD-only analytic cohort and a selected progression outcome from Notebook 02. Notebook 03 adds a baseline predictor matrix and a missingness report.

The most important scientific rule is that all candidate predictors must be baseline-only. Follow-up variables and outcome-derived variables are excluded from the predictor table to reduce data leakage risk.

Variables with high missingness should not automatically be discarded, but they should not be used in the first primary model without a transparent imputation strategy and sensitivity analysis. The next notebook should focus on preprocessing decisions, imputation planning, and train/test splitting strategy, not final modeling.

In [ ]:
# ============================================================
# 15. Write summary report
# ============================================================

summary_lines = []
summary_lines.append("Notebook 03 — Baseline Predictor Table and Missingness Assessment")
summary_lines.append("=" * 72)
summary_lines.append("")
summary_lines.append(f"Input cohort file: {COHORT_FILE}")
summary_lines.append(f"Rows in primary analytic cohort: {len(cohort):,}")
summary_lines.append(f"Unique participants: {cohort['PATNO'].nunique():,}")
summary_lines.append("")
summary_lines.append("Outcome:")
summary_lines.append(f"- Follow-up event(s): {outcome_summary['followup_events']}")
summary_lines.append(f"- Rapid progression positive n: {outcome_summary['rapid_progression_positive_n']:,}")
summary_lines.append(f"- Rapid progression negative n: {outcome_summary['rapid_progression_negative_n']:,}")
summary_lines.append(f"- Rapid progression positive percent: {outcome_summary['rapid_progression_positive_percent']:.2f}%")
summary_lines.append(f"- Annualized delta NP3TOT Q75 threshold: {outcome_summary['annualized_delta_NP3TOT_q75']:.4f}")
summary_lines.append("")
summary_lines.append("Predictor table:")
summary_lines.append(f"- Candidate baseline predictors: {len(predictor_cols_final)}")
summary_lines.append(f"- Predictors with <=30% missingness: {len(low_missing_predictors)}")
summary_lines.append("")
summary_lines.append("Files generated:")
for file_name in [
    "01_required_outcome_column_check.csv",
    "02_outcome_distribution_summary.csv",
    "03_leakage_exclusion_table.csv",
    "04_candidate_predictor_dictionary.csv",
    "05_predictor_missingness_summary.csv",
    "06_outcome_distribution_table.csv",
    "07_feature_matrix_all_candidate_predictors.csv",
    "08_feature_matrix_predictors_missingness_le_30pct.csv",
    "09_baseline_source_event_audit.csv",
    "10_quality_control_checklist.csv",
]:
    summary_lines.append(f"- {file_name}")
summary_lines.append("")
summary_lines.append("Interpretation:")
summary_lines.append("The baseline predictor matrix is ready for scientific review. No imputation, train/test split, or ML modeling has been performed in Notebook 03.")
summary_lines.append("")
summary_lines.append(f"Output folder: {OUTPUT_DIR}")

summary_text = "\n".join(summary_lines)
print(summary_text)

with open(OUTPUT_DIR / "11_notebook_03_summary_report.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

## Expected Output

Notebook 03 should produce the following files:

1. `01_required_outcome_column_check.csv`
2. `02_outcome_distribution_summary.csv`
3. `03_leakage_exclusion_table.csv`
4. `04_candidate_predictor_dictionary.csv`
5. `05_predictor_missingness_summary.csv`
6. `06_outcome_distribution_table.csv`
7. `07_feature_matrix_all_candidate_predictors.csv`
8. `08_feature_matrix_predictors_missingness_le_30pct.csv`
9. `09_baseline_source_event_audit.csv`
10. `10_quality_control_checklist.csv`
11. `11_notebook_03_summary_report.txt`

Do not proceed to modeling until these outputs are reviewed and approved.